# Recording Agent
- Visual understanding
- Performance Tracking
- Debugging
- Evaluation
- Communication

In [4]:
!pip install moviepy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 5.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 16.3 MB/s  0:00:01m0:00:0100:01
  Attempting uninstall: pillow
    Found existing installation: pillow 12.1.1
    Uninstalling pillow-12.1.1:
      Successfully uninstalled pillow-12.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [moviepy]m3/6 [imageio_ffmpeg]


In [5]:
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo
import numpy as np

# Configuration
num_eval_episodes = 4
env_name = "CartPole-v1"  # Replace with your environment

# Create environment with recording capabilities
env = gym.make(env_name, render_mode="rgb_array")  # rgb_array needed for video recording

# Add video recording for every episode
env = RecordVideo(
    env,
    video_folder="cartpole-agent",    # Folder to save videos
    name_prefix="eval",               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
)

# Add episode statistics tracking
env = RecordEpisodeStatistics(env, buffer_length=num_eval_episodes)

print(f"Starting evaluation for {num_eval_episodes} episodes...")
print(f"Videos will be saved to: cartpole-agent/")

for episode_num in range(num_eval_episodes):
    obs, info = env.reset()
    episode_reward = 0
    step_count = 0

    episode_over = False
    while not episode_over:
        # Replace this with your trained agent's policy
        action = env.action_space.sample()  # Random policy for demonstration

        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        step_count += 1

        episode_over = terminated or truncated

    print(f"Episode {episode_num + 1}: {step_count} steps, reward = {episode_reward}")

env.close()

# Print summary statistics
print(f'\nEvaluation Summary:')
print(f'Episode durations: {list(env.time_queue)}')
print(f'Episode rewards: {list(env.return_queue)}')
print(f'Episode lengths: {list(env.length_queue)}')

# Calculate some useful metrics
avg_reward = np.sum(env.return_queue)
avg_length = np.sum(env.length_queue)
std_reward = np.std(env.return_queue)

print(f'\nAverage reward: {avg_reward:.2f} ± {std_reward:.2f}')
print(f'Average episode length: {avg_length:.1f} steps')
print(f'Success rate: {sum(1 for r in env.return_queue if r > 0) / len(env.return_queue):.1%}')

/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /Users/antik/rl/gym/gymnasium/cartpole-agent folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Starting evaluation for 4 episodes...
Videos will be saved to: cartpole-agent/


/Users/antik/anaconda3/envs/gym_env/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Episode 1: 15 steps, reward = 15.0
Episode 2: 12 steps, reward = 12.0
Episode 3: 52 steps, reward = 52.0
Episode 4: 63 steps, reward = 63.0

Evaluation Summary:
Episode durations: [0.011951, 0.008675, 0.04115, 0.044384]
Episode rewards: [15.0, 12.0, 52.0, 63.0]
Episode lengths: [15, 12, 52, 63]

Average reward: 142.00 ± 22.37
Average episode length: 142.0 steps
Success rate: 100.0%


In [ ]:
import logging
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo

# Training configuration
training_period = 250           # Record video every 250 episodes
num_training_episodes = 10_000  # Total training episodes
env_name = "CartPole-v1"

# Set up logging for episode statistics
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Create environment with periodic video recording
env = gym.make(env_name, render_mode="rgb_array")

# Record videos periodically (every 250 episodes)
env = RecordVideo(
    env,
    video_folder="cartpole-training",
    name_prefix="training",
    episode_trigger=lambda x: x % training_period == 0  # Only record every 250th episode
)

# Track statistics for every episode (lightweight)
env = RecordEpisodeStatistics(env)

print(f"Starting training for {num_training_episodes} episodes")
print(f"Videos will be recorded every {training_period} episodes")
print(f"Videos saved to: cartpole-training/")

for episode_num in range(num_training_episodes):
    obs, info = env.reset()
    episode_over = False

    while not episode_over:
        # Replace with your actual training agent
        action = env.action_space.sample()  # Random policy for demonstration
        obs, reward, terminated, truncated, info = env.step(action)
        episode_over = terminated or truncated

    # Log episode statistics (available in info after episode ends)
    if "episode" in info:
        episode_data = info["episode"]
        logging.info(f"Episode {episode_num}: "
                    f"reward={episode_data['r']:.1f}, "
                    f"length={episode_data['l']}, "
                    f"time={episode_data['t']:.2f}s")

        # Additional analysis for milestone episodes
        if episode_num % 1000 == 0:
            # Look at recent performance (last 100 episodes)
            recent_rewards = list(env.return_queue)[-100:]
            if recent_rewards:
                avg_recent = sum(recent_rewards) / len(recent_rewards)
                print(f"  -> Average reward over last 100 episodes: {avg_recent:.1f}")

env.close()

"""
     ---                                                                                                                                                  
  1. 配置部分
                                                                                                                                                       
  training_period = 250           # 每250个episode录制一次视频
  num_training_episodes = 10_000  # 总共训练10000个episode
  env_name = "CartPole-v1"
  - training_period=250：每250个episode才录制视频（避免视频文件太多）
  - num_training_episodes=10000：训练总量

  ---
  2. 环境创建

  env = gym.make(env_name, render_mode="rgb_array")
  env = RecordVideo(...)  # 录制视频
  env = RecordEpisodeStatistics(env)  # 记录统计数据

  三个 wrapper 的作用：

  ┌─────────────────────────┬──────────────────────────────────────────┐
  │         Wrapper         │                   功能                   │
  ├─────────────────────────┼──────────────────────────────────────────┤
  │ RecordVideo             │ 保存训练视频到 cartpole-training/ 文件夹 │
  ├─────────────────────────┼──────────────────────────────────────────┤
  │ RecordEpisodeStatistics │ 记录每个 episode 的 reward、length、time │
  └─────────────────────────┴──────────────────────────────────────────┘

  ---
  3. 训练循环

  for episode_num in range(num_training_episodes):
      obs, info = env.reset()
      episode_over = False

      while not episode_over:
          action = env.action_space.sample()  # 随机动作
          obs, reward, terminated, truncated, info = env.step(action)
          episode_over = terminated or truncated

  - env.reset()：重置环境，返回初始观察
  - env.step(action)：执行动作，返回 (obs, reward, terminated, truncated, info)
  - terminated or truncated：任一为 True 表示 episode 结束

  ---
  4. 获取统计数据

  if "episode" in info:
      episode_data = info["episode"]
      logging.info(f"Episode {episode_num}: "
                  f"reward={episode_data['r']:.1f}, "
                  f"length={episode_data['l']}, "
                  f"time={episode_data['t']:.2f}s")

  info["episode"] 包含：

  ┌─────┬──────────────────────┐
  │ 键  │         含义         │
  ├─────┼──────────────────────┤
  │ r   │ 总 reward            │
  ├─────┼──────────────────────┤
  │ l   │ episode 长度（步数） │
  ├─────┼──────────────────────┤
  │ t   │ episode 耗时（秒）   │
  └─────┴──────────────────────┘

  ---
  5. 周期性性能检查

  if episode_num % 1000 == 0:
      recent_rewards = list(env.return_queue)[-100:]
      avg_recent = sum(recent_rewards) / len(recent_rewards)

  - 每 1000 个 episode 查看最近 100 个 episode 的平均 reward
  - 用于监控训练进度

  ---
  总结

  这是一个训练框架模板，当前使用随机策略（sample()）。实际使用时只需替换这一行：

  action = your_trained_agent.select_action(obs)  # 替换为你的智能体

  即可开始真正训练。
"""